In [1]:
%pip install --user numpy pandas scikit-learn joblib

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import os


# Membaca file asli menggunakan sep=';' karena data kamu dipisahkan oleh titik koma
df = pd.read_csv('DataSet_Lari.csv', sep=';')

print(f"Dataset asli berhasil dimuat! Total: {df.shape[0]} baris data.")
print("\n--- Ringkasan Kolom & Tipe Data Mentah ---")
df.info()
df.head()

Dataset asli berhasil dimuat! Total: 42116 baris data.

--- Ringkasan Kolom & Tipe Data Mentah ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42116 entries, 0 to 42115
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   athlete                   42116 non-null  int64  
 1   gender                    41761 non-null  object 
 2   timestamp                 42116 non-null  object 
 3   distance (m)              42116 non-null  float64
 4   elapsed time (s)          42116 non-null  int64  
 5   elevation gain (m)        42116 non-null  float64
 6   average heart rate (bpm)  23732 non-null  float64
dtypes: float64(3), int64(2), object(2)
memory usage: 2.2+ MB


,athlete,gender,timestamp,distance (m),elapsed time (s),elevation gain (m),average heart rate (bpm)
0,18042525,M,15/12/2019 09:08,2965.8,812,17.4,150.3
1,18042525,M,10/12/2019 19:27,10020.8,3290,52.2,160.8
2,18042525,M,03/12/2019 19:46,12132.2,4027,249.0,148.9
3,18042525,M,26/11/2019 19:46,11631.5,4442,194.0,136.2
4,18042525,M,19/11/2019 19:45,11708.1,4022,250.7,146.0


In [3]:


# 1. Hitung kecepatan tiap sesi untuk menyaring data yang tidak logis
df['speed_session'] = df['distance (m)'] / df['elapsed time (s)']

# 2. Saringan ketat: Jarak & Waktu harus > 0, Kecepatan lari manusia normal (1 m/s sampai 10 m/s)
kondisi_valid = (
    (df['distance (m)'] > 0) & 
    (df['elapsed time (s)'] > 0) & 
    (df['speed_session'] >= 1.0) & 
    (df['speed_session'] <= 10.0)
)
df_clean = df[kondisi_valid].copy()

# 3. Buang baris yang kolom 'gender'-nya kosong (missing values)
df_clean = df_clean.dropna(subset=['gender'])

# Hapus kolom pembantu speed_session agar tidak mencontek target
df_clean = df_clean.drop(columns=['speed_session'])

# 4. Ekstrak Fitur Waktu dari kolom 'timestamp' asli
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'], format='%d/%m/%Y %H:%M')
df_clean['Jam'] = df_clean['timestamp'].dt.hour

def tentukan_waktu(jam):
    if 5 <= jam < 11: return 'Pagi'
    elif 11 <= jam < 16: return 'Siang'
    else: return 'Malam'

df_clean['Waktu_Lari'] = df_clean['Jam'].apply(tentukan_waktu)

print(f"Proses Cleaning Selesai! Data bersih siap pakai: {df_clean.shape[0]} baris.")
df_clean.head()

Proses Cleaning Selesai! Data bersih siap pakai: 41137 baris.


,athlete,gender,timestamp,distance (m),elapsed time (s),elevation gain (m),average heart rate (bpm),Jam,Waktu_Lari
0,18042525,M,2019-12-15 09:08:00,2965.8,812,17.4,150.3,9,Pagi
1,18042525,M,2019-12-10 19:27:00,10020.8,3290,52.2,160.8,19,Malam
2,18042525,M,2019-12-03 19:46:00,12132.2,4027,249.0,148.9,19,Malam
3,18042525,M,2019-11-26 19:46:00,11631.5,4442,194.0,136.2,19,Malam
4,18042525,M,2019-11-19 19:45:00,11708.1,4022,250.7,146.0,19,Malam


In [4]:


# 1. Hitung kecepatan lari rata-rata historis per ID Atlet
df_clean['speed'] = df_clean['distance (m)'] / df_clean['elapsed time (s)']
athlete_data = df_clean.groupby('athlete')['speed'].mean().reset_index()

# 2. Standardisasi skala kecepatan atlet
scaler_cluster = StandardScaler()
athlete_data['speed_scaled'] = scaler_cluster.fit_transform(athlete_data[['speed']])

# 3. Jalankan K-Means membagi 116 atlet ke dalam 3 kelas keahlian
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
athlete_data['cluster'] = kmeans.fit_predict(athlete_data[['speed_scaled']])

# 4. Urutkan kelompok agar konsisten: 0=Beginner, 1=Intermediate, 2=Advanced
urutan_kecepatan = athlete_data.groupby('cluster')['speed'].mean().sort_values().index
mapping_urutan = {urutan_kecepatan[0]: 0, urutan_kecepatan[1]: 1, urutan_kecepatan[2]: 2}
athlete_data['cluster'] = athlete_data['cluster'].map(mapping_urutan)

# 5. Petakan hasil cluster dari tabel atlet kembali ke tabel utama (42.116 baris)
athlete_cluster_map = dict(zip(athlete_data['athlete'], athlete_data['cluster']))
df_clean['Cluster_Label'] = df_clean['athlete'].map(athlete_cluster_map)

# Ubah angka menjadi kategori teks
df_clean['Tingkat_Pengalaman'] = df_clean['Cluster_Label'].map({0: 'Beginner', 1: 'Intermediate', 2: 'Advanced'})

print("Sebaran data tingkat pengalaman pelari yang berhasil dipetakan secara seimbang (balance):")
print(df_clean['Tingkat_Pengalaman'].value_counts())

Sebaran data tingkat pengalaman pelari yang berhasil dipetakan secara seimbang (balance):
Tingkat_Pengalaman
Intermediate    22615
Beginner        10678
Advanced         7844
Name: count, dtype: int64


In [ ]:
print("--- Cell 4: Encoding, Training Regresi, & Simpan Model ---")

# 1. MANUAL ENCODING (Mengubah teks kategori menjadi angka biner 0 atau 1)
# Kolom Gender: M menjadi 1, F menjadi 0
df_clean['gender_M'] = (df_clean['gender'] == 'M').astype(int)

# Kolom Waktu Lari: Dipecah menjadi Pagi dan Siang (Jika Malam, keduanya otomatis 0)
df_clean['Waktu_Lari_Pagi'] = (df_clean['Waktu_Lari'] == 'Pagi').astype(int)
df_clean['Waktu_Lari_Siang'] = (df_clean['Waktu_Lari'] == 'Siang').astype(int)

# Kolom Tingkat Pengalaman (Hasil Klasterisasi Cell 3): Dipecah menjadi Intermediate dan Advanced
df_clean['Tingkat_Pengalaman_Intermediate'] = (df_clean['Tingkat_Pengalaman'] == 'Intermediate').astype(int)
df_clean['Tingkat_Pengalaman_Advanced'] = (df_clean['Tingkat_Pengalaman'] == 'Advanced').astype(int)

# 2. DEFINISIKAN FITUR INPUT (X) DAN TARGET OUTPUT (y)
# Sesuai dengan rencana parameter yang dimasukkan user di aplikasi web RunPace
fitur_X_manual = [
    'distance (m)', 'elevation gain (m)', 'gender_M', 
    'Waktu_Lari_Pagi', 'Waktu_Lari_Siang',
    'Tingkat_Pengalaman_Intermediate', 'Tingkat_Pengalaman_Advanced'
]

X_manual = df_clean[fitur_X_manual]
y_manual = df_clean['elapsed time (s)']

# 3. TRAIN-TEST SPLIT (Membagi data: 80% Latihan, 20% Ujian Akhir)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_manual, y_manual, test_size=0.2, random_state=42
)

print(f"Data berhasil dibelah:")
print(f" - Jumlah Data Training : {X_train_m.shape[0]} baris")
print(f" - Jumlah Data Testing  : {X_test_m.shape[0]} baris")

# 4. TRAINING MODEL REGRESI (Random Forest Regressor)
print("\nMelatih model Random Forest Regressor (Mohon tunggu)...")
# n_estimators=30 artinya menggunakan 30 pohon keputusan, n_jobs=-1 agar MacBook Pro kamu ngebut pake semua core
model_regresi_m = RandomForestRegressor(n_estimators=30, random_state=42, n_jobs=-1)
model_regresi_m.fit(X_train_m, y_train_m)
print("-> Proses Training Selesai!")

# 5. EVALUASI AKURASI MODEL (Melihat Nilai Rapor)
y_pred_m = model_regresi_m.predict(X_test_m)
mae_manual = mean_absolute_error(y_test_m, y_pred_m)
r2_manual = r2_score(y_test_m, y_pred_m)

print(f"\n✨ RAPOR EVALUASI AKURASI MODEL:")
print(f" - Mean Absolute Error (MAE) : {mae_manual:.2f} detik")
print(f" - R-Squared (R2) Score      : {r2_manual * 100:.2f}%")

# 6. SAVE MODEL (Membekukan Otak AI Menjadi File .pkl)
# Membuat folder 'models' otomatis jika belum ada di direktori Mac kamu
if not os.path.exists('models'):
    os.makedirs('models')

# Menyimpan file model utama dan cetakan nama fiturnya
joblib.dump(model_regresi_m, 'models/runpace_regressor.pkl')
joblib.dump(fitur_X_manual, 'models/model_features.pkl')

print("\n🎉 KEMENANGAN! File 'runpace_regressor.pkl' dan 'model_features.pkl' sukses terbit di folder 'models'.")

--- Cell 4: Encoding, Training Regresi, & Simpan Model ---
Data berhasil dibelah:
 - Jumlah Data Training : 32909 baris
 - Jumlah Data Testing  : 8228 baris

Melatih model Random Forest Regressor (Mohon tunggu)...
-> Proses Training Selesai!

✨ RAPOR EVALUASI AKURASI MODEL:
 - Mean Absolute Error (MAE) : 460.03 detik
 - R-Squared (R2) Score      : 90.25%

🎉 KEMENANGAN! File 'runpace_regressor.pkl' dan 'model_features.pkl' sukses terbit di folder 'models'.
